<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-03-analyze-fleet-in-ghanaian-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze a fleet in Ghanaian EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Monitor a Fleet (a group of vessels) of Tuna Longliners in [Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400) region for Compliance** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**, **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)**, and **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import datetime
import os

import pandas as pd

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: Monitoring a Fleet of Tuna Longliners for Compliance**

Kwame is a fisheries compliance officer in Ghana, responsible for monitoring a fleet of tuna longliners operating within **[Ghanaian Exclusive Economic Zone (EEZ)](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**. His goal is to:

1. Track apparent fishing effort for **longliners** over the last 12 months.
2. Identify potential vessels in this fleet, their operational patterns, and their activity levels.
3. Retrieve vessel details, including **flag state**, **ownership history**, and **authorizations**.
4. Analyze events such as **port visits** and **encounters (potential transshipment)** activities.

**APIs Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Retrieve apparent fishing effort grouped by vessel ID in Ghanaian EEZ.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** – Get vessel identity, ownership, and compliance details.
3. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** – Identify port visits and potential transshipment activities for vessels in the fleet.

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)
- [Vessel ID](https://globalfishingwatch.org/our-apis/documentation#vessel-id)
- [Vessel API - Vessel identity information](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Ghanaian EEZ

Before making API requests, Kwame must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Ghanaian EEZ, the region ID is 8400](https://www.marineregions.org/gazetteer.php?p=details&id=8400)** (public-eez-areas dataset).

**Note:** See how to use the [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) to obtain and filter predefined [**Regions of Interest (ROIs)**](https://globalfishingwatch.org/our-apis/documentation#regions), such as Exclusive Economic Zones (**EEZs**), Marine Protected Areas (**MPAs**), and Regional Fisheries Management Organizations (**RFMOs**).

In [5]:
eez_rois_result = await gfw_client.references.get_eez_regions(iso3="GHA")
gha_eez_roi = eez_rois_result.data()[0]

In [6]:
gha_eez_roi.id, gha_eez_roi.dataset, gha_eez_roi.label, gha_eez_roi.iso3

('8400', 'public-eez-areas', 'Ghanaian Exclusive Economic Zone', 'GHA')

## Step 1: Retrieve Fishing Effort in Ghanaian EEZ

Kwame **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **fishing effort for all vessels**, grouping them by **gear type** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - 8400 Ghanaian EEZ
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 12 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Gear Type

In [7]:
end_date = datetime.date.today()

In [8]:
start_date = end_date - datetime.timedelta(weeks=52)

In [9]:
start_date, end_date

(datetime.date(2025, 6, 27), datetime.date(2026, 6, 26))

In [10]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    group_by="GEARTYPE",
    temporal_resolution="ENTIRE",
    start_date=start_date,  # "2024-01-01"
    end_date=end_date,  # "2025-01-01"
    spatial_aggregation=True,
    region=gha_eez_roi,
)

In [11]:
step_1_report_df = step_1_report_result.df()

In [12]:
step_1_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     7 non-null      str    
 1   detections               0 non-null      object 
 2   flag                     0 non-null      object 
 3   gear_type                7 non-null      str    
 4   hours                    7 non-null      float64
 5   vessel_ids               7 non-null      int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign                0 non-null      

In [13]:
step_1_report_df[["flag", "gear_type", "hours", "vessel_ids"]].head()

,flag,gear_type,hours,vessel_ids
0,None,drifting_longlines,663.434444,2
1,None,fishing,23077.302778,30
2,None,inconclusive,22606.803611,27
3,None,other_purse_seines,284.236944,1
4,None,pole_and_line,2955.103611,3


In [14]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["gear_type"], as_index=False)
    .agg(hours=("hours", "sum"), vessel_ids=("vessel_ids", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [15]:
step_1_agg_report_df

,gear_type,hours,vessel_ids
1,fishing,23077.302778,30
2,inconclusive,22606.803611,27
5,trawlers,22540.676667,29
6,tuna_purse_seines,5324.287778,20
4,pole_and_line,2955.103611,3
0,drifting_longlines,663.434444,2
3,other_purse_seines,284.236944,1


### What We have Learned from Step 1

1. Kwame now has apparent fishing effort data for multiple gear types.
2. There are **potential 2 vessels** operating as a **longliners (i.e., drifting_longlines)** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)** with `663.434444 hours` logged.

## Step 2: Retrieve Vessel IDs for Longliners

Kwame refines his **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** request to group by **vessel ID**, and filtering only for **longliners** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**.

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - [8400 Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 12 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Vessel ID 

**Why Use group-by=VESSEL_ID?**

Grouping by **VESSEL_ID** allows **individual vessel identification** in the response. This is crucial for **tracking vessel activity** and, more importantly, linking each detected vessel to the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** in the next step. By structuring the query this way, we can fetch vessel details such as **flag, name, and ownership records** in **Step 3 below**.


In [16]:
step_2_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    group_by="VESSEL_ID",
    temporal_resolution="ENTIRE",
    filters=["geartype in ('drifting_longlines')"],
    start_date=start_date,
    end_date=end_date,
    spatial_aggregation=True,
    region=gha_eez_roi,
)

In [17]:
step_2_report_df = step_2_report_result.df()

In [18]:
step_2_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   date                     2 non-null      str                
 1   detections               0 non-null      object             
 2   flag                     2 non-null      str                
 3   gear_type                2 non-null      str                
 4   hours                    2 non-null      float64            
 5   vessel_ids               0 non-null      object             
 6   vessel_id                2 non-null      str                
 7   vessel_type              2 non-null      str                
 8   entry_timestamp          2 non-null      datetime64[us, UTC]
 9   exit_timestamp           2 non-null      datetime64[us, UTC]
 10  first_transmission_date  2 non-null      datetime64[us, UTC]
 11  last_transmission_date   2 non-null      dateti

In [19]:
step_2_report_df[["flag", "gear_type", "hours", "mmsi", "ship_name"]].head()

,flag,gear_type,hours,mmsi,ship_name
0,TWN,DRIFTING_LONGLINES,2.213056,416004904,MAAN FARN NO.1
1,,DRIFTING_LONGLINES,661.221389,983110470,


### What We have Learned from Step 2

1. Kwame identifies `2 vessels` operating as longliners within **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**.
2. The vessel `(mmsi: 983110470)` shows significant activity with `661.221389 hours` logged.
3. Other vessels `(mmsi: 416004904)` shows apparent fishing effort over a short duration.
4. This response is based on **AIS self-reported** data and should be further validated.


## Step 3: Retrieve Vessel Details Using the Vessels API

Kwame queries the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** to **get detailed vessel identity and ownership records**. Please [learn more about Vessels API here](https://globalfishingwatch.org/our-apis/documentation#vessels-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information).

**Filters Used:**

1. **Vessel IDs** from 4Wings API, **Step 2 above**.
2. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)** - `public-global-vessel-identity:latest`.
3. **[Includes](https://globalfishingwatch.org/our-apis/documentation#get-vessels-by-ids-url-parameters)** - `POTENTIAL_RELATED_SELF_REPORTED_INFO`.

**Note:** Vessels may change identifiers over time, such as their `Maritime Mobile Service Identity (MMSI)`,` International Maritime Organization (IMO) number)`, `call sign`, or even their `name`. These changes can occur due to `re-registration`, `changes in ownership`, or other `operational reasons` within the `AIS transponder`. Parameter (`includes = POTENTIAL_RELATED_SELF_REPORTED_INFO`) helps group all **vessel ids** that are **potentially related** as part of the **same physical vessel** based on publicly available registry information.

In [20]:
step_2_vessel_ids = list(step_2_report_df["vessel_id"].unique())

In [21]:
step_2_vessel_ids

['dc4940890-0883-9dd7-797b-ce8edfc33b2d',
 '5ad9cc284-44ca-8033-43f5-8ce653b05834']

In [22]:
step_3_vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=step_2_vessel_ids,
)

In [23]:
step_3_vessels_df = step_3_vessels_result.df()

In [24]:
step_3_vessels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         2 non-null      str   
 1   registry_info_total_records     2 non-null      int64 
 2   registry_info                   2 non-null      object
 3   registry_owners                 2 non-null      object
 4   registry_public_authorizations  2 non-null      object
 5   combined_sources_info           2 non-null      object
 6   self_reported_info              2 non-null      object
dtypes: int64(1), object(5), str(1)
memory usage: 244.0+ bytes


**Understanding Vessel Details Response Data**

- **registryInfoTotalRecords** – This represents the **number of registry records** found for the vessels.
- **registryInfo** – Contains **public registry data**. This data is sourced from official **vessel registries**.
- **registryOwners** – Lists the **registered owners** of the vessel based on public sources.
- **registryPublicAuthorizations** – Represents known **fishing authorizations** from public sources. Users should verify against national registries and RFMO records for additional context.
- **combinedSourcesInfo** – Provides inferred data from multiple sources, including. This is not explicitly reported by vessels but determined through **GFW's classification methods**.
- **selfReportedInfo** – Contains **AIS self-reported** data, including `MMSI`, `ship name`, and `flag` as broadcast by the **vessel itself**. Self-reported data may not always align with registry data and should be cross-checked.

In [25]:
step_3_vessels_df[["registry_info", "registry_owners", "self_reported_info"]]

,registry_info,registry_owners,self_reported_info
0,"[{'id': '05bb903aecec1cc5efc55708d7fd0868', 's...","[{'name': 'HASBRO FISHERIES GROUP', 'flag': 'T...",[{'id': 'dc4940890-0883-9dd7-797b-ce8edfc33b2d...
1,[],[],[{'id': '5ad9cc284-44ca-8033-43f5-8ce653b05834...


### Explore Vessels Registry Info

In [26]:
step_3_has_registry_info_mask = step_3_vessels_df[
    "registry_info"
].notna() & step_3_vessels_df["registry_info"].astype(bool)

In [27]:
step_3_registry_info_df = pd.json_normalize(
    step_3_vessels_df[step_3_has_registry_info_mask]["registry_info"].explode()
)

In [28]:
step_3_registry_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "gear_types", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,gear_types,source_code
0,416004904,TWN,MAAN FARN 1,MAANFARN1,[DRIFTING_LONGLINES],"[ICCAT, IMO, ISSF, OPRT, RESEARCH-PAPER, SNP]"


### Explore Registry Owners

In [29]:
step_3_has_registry_owners_mask = step_3_vessels_df[
    "registry_owners"
].notna() & step_3_vessels_df["registry_owners"].astype(bool)

In [30]:
step_3_registry_owners_df = pd.json_normalize(
    step_3_vessels_df[step_3_has_registry_owners_mask]["registry_owners"].explode()
)

In [31]:
step_3_registry_owners_match_registry_info_mask = step_3_registry_owners_df[
    "ssvid"
].isin(step_3_registry_info_df["ssvid"])

In [32]:
step_3_registry_owners_df[step_3_registry_owners_match_registry_info_mask][
    ["ssvid", "flag", "name", "source_code"]
]

,ssvid,flag,name,source_code
0,416004904,TWN,HASBRO FISHERIES GROUP,[RESEARCH-PAPER]


### Explore Vessels Self Reported Info

In [33]:
step_3_has_self_reported_info_mask = step_3_vessels_df[
    "self_reported_info"
].notna() & step_3_vessels_df["self_reported_info"].astype(bool)

In [34]:
step_3_self_reported_info_df = pd.json_normalize(
    step_3_vessels_df[step_3_has_self_reported_info_mask][
        "self_reported_info"
    ].explode()
)

In [35]:
step_3_self_reported_info_df[
    [
        "ssvid",
        "flag",
        "ship_name",
        "n_ship_name",
        "source_code",
        "transmission_date_from",
    ]
]

,ssvid,flag,ship_name,n_ship_name,source_code,transmission_date_from
0,416004904,TWN,MAAN FARN NO.1,MAANFARN1,[AIS],2017-09-24 12:55:55+00:00
0,416004904,TWN,MAAN FAKN NO.1,MAANFAKN1,[AIS],2015-04-22 01:47:27+00:00
1,983110470,NaN,NaN,NaN,[AIS],2022-11-16 11:32:26+00:00


### What We have Learned from Step 3

- The vessel `mmsi/ssvid: 983110470` appears to be a drifting longliner flagged under China.
- No public registry data is found for this vessel.
- The vessel's identity information is based on `AIS self-reported data`, which may not always align with official registries.
- The vessel appears to have been active since `2022`, based on `self-reported AIS records`.
- This vessel's data needs further validation against official public sources

## Step 4: Detect Fleet Activity (Port Visits)

Now that Kwame has identified vessels in the fleet, he examines their activity further by querying the **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**. This allows him to detect `port visits`, `encounters (Potential Transshipment)` and `apparent fishing activity` based on vessel movement patterns. Please [learn more about Events API here](https://globalfishingwatch.org/our-apis/documentation#events-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#how-are-the-events-estimated).

**Filters Used:**

1. **Vessel ID** from 4Wings API
2. **[Event Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - Port visits, encounters (potential transshipment), and fishing events.
3. **Time Range** - Last 12 months.
4. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)**:
   - `public-global-port-visits-events::latest` (Port Visits)
   - `public-global-encounters-events:latest` (Encounters between vessels)
   - `public-global-fishing-events:latest` (Fishing activity)
5. **[Encounter Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - FISHING-FISHING

In [36]:
step_4_events_result = await gfw_client.events.get_all_events(
    datasets=[
        "public-global-encounters-events:latest",
        "public-global-fishing-events:latest",
        "public-global-port-visits-events:latest",
    ],
    vessels=step_2_vessel_ids,
    types=["ENCOUNTER", "FISHING", "PORT_VISIT"],
    start_date=start_date,
    end_date=end_date,
    encounter_types=["FISHING-FISHING"],
)

In [37]:
step_4_events_df = step_4_events_result.df()

In [38]:
step_4_events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         501 non-null    datetime64[us, UTC]
 1   end           501 non-null    datetime64[us, UTC]
 2   id            501 non-null    str                
 3   type          501 non-null    str                
 4   position      501 non-null    object             
 5   regions       501 non-null    object             
 6   bounding_box  501 non-null    object             
 7   distances     501 non-null    object             
 8   vessel        501 non-null    object             
 9   encounter     0 non-null      object             
 10  fishing       483 non-null    object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    18 non-null     object             
dtypes: datetime64[us, UTC

In [39]:
step_4_events_df["type"].value_counts()

type
fishing       483
port_visit     18
Name: count, dtype: int64

### Explore Apparent Fishing Events

In [40]:
step_4_fishing_events_df = step_4_events_df[step_4_events_df["fishing"].notna()]

In [41]:
step_4_fishing_df = pd.concat(
    [
        pd.json_normalize(step_4_fishing_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_fishing_events_df["fishing"], sep="_"),
    ],
    axis=1,
)

In [42]:
step_4_fishing_df.info()

<class 'pandas.DataFrame'>
Index: 483 entries, 0 to 493
Data columns (total 12 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id                                  483 non-null    str    
 1   name                                460 non-null    str    
 2   ssvid                               483 non-null    str    
 3   flag                                460 non-null    str    
 4   type                                483 non-null    str    
 5   public_authorizations               483 non-null    object 
 6   nextPort                            0 non-null      object 
 7   total_distance_km                   483 non-null    float64
 8   average_speed_knots                 483 non-null    float64
 9   average_duration_hours              0 non-null      object 
 10  potential_risk                      483 non-null    bool   
 11  vessel_public_authorization_status  483 non-null    str    
d

In [43]:
step_4_fishing_df[
    [
        "name",
        "ssvid",
        "total_distance_km",
        "average_speed_knots",
    ]
]

,name,ssvid,total_distance_km,average_speed_knots
0,NaN,983110470,199.845790,4.646491
1,MAAN FARN NO.1,416004904,30.629027,4.584615
2,MAAN FARN NO.1,416004904,36.941824,8.577778
3,NaN,983110470,21.235980,4.292857
4,NaN,983110470,30.004830,5.738462
...,...,...,...,...
489,MAAN FARN NO.1,416004904,2.436448,3.015385
490,MAAN FARN NO.1,416004904,11.147134,2.141176
491,MAAN FARN NO.1,416004904,6.077442,1.147541
492,MAAN FARN NO.1,416004904,39.414255,3.263636


In [44]:
step_4_fishing_df["ssvid"].value_counts()

ssvid
416004904    460
983110470     23
Name: count, dtype: int64

### Explore Port Visit Events

In [45]:
step_4_port_visit_events_df = step_4_events_df[step_4_events_df["port_visit"].notna()]

In [46]:
step_4_port_visits_df = pd.concat(
    [
        pd.json_normalize(step_4_port_visit_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_port_visit_events_df["port_visit"], sep="_"),
    ],
    axis=1,
)

In [47]:
step_4_port_visits_df.info()

<class 'pandas.DataFrame'>
Index: 18 entries, 25 to 500
Data columns (total 37 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id                                             18 non-null     str    
 1   name                                           3 non-null      str    
 2   ssvid                                          18 non-null     str    
 3   flag                                           3 non-null      str    
 4   type                                           18 non-null     str    
 5   public_authorizations                          18 non-null     object 
 6   nextPort                                       0 non-null      object 
 7   visit_id                                       18 non-null     str    
 8   confidence                                     18 non-null     str    
 9   duration_hrs                                   18 non-null     float64

In [48]:
step_4_port_visits_df[
    [
        "name",
        "ssvid",
        "confidence",
        "start_anchorage_name",
        "intermediate_anchorage_name",
        "end_anchorage_name",
    ]
]

,name,ssvid,confidence,start_anchorage_name,intermediate_anchorage_name,end_anchorage_name
25,NaN,983110470,4,NaN,NaN,NaN
66,NaN,983110470,4,KIZOMBA FPSO,KIZOMBA FPSO,KIZOMBA FPSO
67,NaN,983110470,4,LAGOS,LAGOS,LAGOS
68,NaN,983110470,4,BLOCK 15,BLOCK 15,BLOCK 15
69,NaN,983110470,4,NaN,NaN,NaN
70,NaN,983110470,4,LUANDA,LUANDA,LUANDA
71,NaN,983110470,4,COD-4,COD-4,COD-4
72,NaN,983110470,4,NaN,NaN,NaN
365,NaN,983110470,4,KIZOMBA FPSO,KIZOMBA FPSO,BLOCK 15
366,NaN,983110470,4,KIZOMBA FPSO,KIZOMBA FPSO,KIZOMBA FPSO


### What We’ve Learned from Step 4

- `18: port visits`, `0: encounters`, and `484: fishing` events were found for the queried vessels in the given date range.
- Some events were missed due to **AIS data coverage gaps**.
- Different filters may need to be applied to refine results.

**Caveats & Considerations**

- **A lack of recorded encounters or any other events does not confirm the absence of such activities**—AIS coverage, reporting behavior, and dataset updates can impact results.
- **Further investigation may be required**, including manual validation using historical data or consulting additional sources.

## Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Identify fishing effort by **gear type** in Ghanaian EEZ.
2. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve vessel IDs for potential **longliners**.
3. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** - Fetch detailed **vessel identity** & **ownership**.
4. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** - Attempt to detect fleet activity (**port visits**, **encounters**, and **apparent fishing** events)